# Stage 0 — COSMOS Manual Cutout Preparation with Manifest

**Purpose:** Build a clean source-galaxy library from **manually downloaded** COSMOS HST/ACS F814W FITS cutouts while still using **catalog-based rejection** through a CSV manifest.

## What this notebook does
1. Reads a **manifest CSV** that stores catalog information for each manually downloaded cutout.
2. Applies **catalog-based rejection** using fields like `mu_class`, `flags`, `flux_radius`, `elongation`, and `mag_auto`.
3. Matches accepted manifest rows to locally downloaded FITS files.
4. Performs image-level QA, normalisation, and `.npy` export.
5. Writes metadata and a grid JPG for review.

## Why this version is useful
This keeps the **download step manual** while preserving the scientifically important **catalog selection logic** from the original prototype notebook.

## Cell 1 — Install dependencies (run once)

In [ ]:
import subprocess
import sys

packages = [
    'numpy',
    'pandas',
    'matplotlib',
    'astropy',
    'tqdm',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('Dependencies installed successfully.')

## Cell 2 — Imports and configuration

A **manifest** is a small table, usually CSV, where each row describes one cutout you downloaded.

Typical columns:
- `filename`: FITS filename on disk
- `ra`, `dec`: sky position
- `mu_class`: star/galaxy class from catalog
- `flags`: detection-quality flag
- `flux_radius`: size estimate
- `elongation`: axis ratio proxy
- `mag_auto`: brightness estimate

This notebook assumes you already downloaded the FITS files manually and also prepared this manifest CSV.

In [ ]:
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales

warnings.filterwarnings('ignore')

# =========================
# User configuration
# =========================
RAW_DIR = Path('cosmos_cutouts_manual/raw_fits')
OUT_DIR = Path('cosmos_cutouts_manual_with_manifest')
NPY_DIR = OUT_DIR / 'npy'
META_DIR = OUT_DIR / 'metadata'
PLOT_DIR = OUT_DIR / 'qa_plots'
MANIFEST_PATH = Path('cosmos_cutouts_manual/manual_cutout_manifest.csv')

for d in [RAW_DIR, OUT_DIR, NPY_DIR, META_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CUTOUT_SIZE_ARCSEC = 10.0
DEFAULT_PIXEL_SCALE_ARCSEC = 0.03
EXPECTED_SIZE_PIX = int(round(CUTOUT_SIZE_ARCSEC / DEFAULT_PIXEL_SCALE_ARCSEC))

# Catalog-based rejection thresholds
KEEP_ONLY_MU_CLASS_1 = True
REQUIRE_FLAGS_ZERO = True
MAG_AUTO_RANGE = (20.0, 27.0)
FLUX_RADIUS_RANGE = (2.5, 40.0)
ELONGATION_RANGE = (1.0, 4.0)

# Manual-review controls
REJECT_BY_FILENAME = set()
MAX_ROWS = None

# Simple image-level QA thresholds
MIN_STD = 0.003
MAX_PEAK_AFTER_NORM = 20.0
MAX_EDGE_ABS_MEAN = 0.50
MIN_FINITE_FRACTION = 0.98

print(f'RAW_DIR                : {RAW_DIR.resolve()}')
print(f'MANIFEST_PATH          : {MANIFEST_PATH.resolve()}')
print(f'OUT_DIR                : {OUT_DIR.resolve()}')
print(f'Cutout size            : {CUTOUT_SIZE_ARCSEC:.1f}"')
print(f'Default pixel scale    : {DEFAULT_PIXEL_SCALE_ARCSEC:.3f}"/pixel')
print(f'Expected cutout size   : ~{EXPECTED_SIZE_PIX} px per side')

## Cell 3 — Load and validate the manifest

The manifest is your bridge between **manual files** and **catalog metadata**.

Without it, the notebook only knows there is a FITS file. With it, the notebook knows whether that file corresponds to a likely galaxy, whether the detection was flagged, how large the source is, and how bright it is.

In [ ]:
required_columns = [
    'filename',
    'ra',
    'dec',
    'mu_class',
    'flags',
    'flux_radius',
    'elongation',
    'mag_auto',
]

manifest_df = pd.read_csv(MANIFEST_PATH)

if MAX_ROWS is not None:
    manifest_df = manifest_df.head(MAX_ROWS).copy()

missing = [c for c in required_columns if c not in manifest_df.columns]
if missing:
    raise ValueError(f'Manifest is missing required columns: {missing}')

for col in ['ra', 'dec', 'mu_class', 'flags', 'flux_radius', 'elongation', 'mag_auto']:
    manifest_df[col] = pd.to_numeric(manifest_df[col], errors='coerce')

print('Manifest rows:', len(manifest_df))
display(manifest_df.head())

## Cell 4 — Apply catalog-based rejection

**Catalog-based rejection** means rejecting objects using survey catalog measurements instead of relying only on the image cutout.

Examples:
- `mu_class != 1` can reject likely stars.
- `flags != 0` can reject questionable detections.
- extreme `flux_radius` can reject pathological size measurements.
- extreme `elongation` can reject odd or problematic shapes.
- very bright or very faint `mag_auto` can remove problematic sources.

In [ ]:
filtered = manifest_df.copy()
filtered['catalog_reject_reason'] = ''

if KEEP_ONLY_MU_CLASS_1:
    mask = filtered['mu_class'] != 1
    filtered.loc[mask, 'catalog_reject_reason'] += 'mu_class_not_1;'

if REQUIRE_FLAGS_ZERO:
    mask = filtered['flags'] != 0
    filtered.loc[mask, 'catalog_reject_reason'] += 'flags_nonzero;'

mask = ~filtered['mag_auto'].between(MAG_AUTO_RANGE[0], MAG_AUTO_RANGE[1], inclusive='both')
filtered.loc[mask, 'catalog_reject_reason'] += 'mag_auto_out_of_range;'

mask = ~filtered['flux_radius'].between(FLUX_RADIUS_RANGE[0], FLUX_RADIUS_RANGE[1], inclusive='both')
filtered.loc[mask, 'catalog_reject_reason'] += 'flux_radius_out_of_range;'

mask = ~filtered['elongation'].between(ELONGATION_RANGE[0], ELONGATION_RANGE[1], inclusive='both')
filtered.loc[mask, 'catalog_reject_reason'] += 'elongation_out_of_range;'

mask = filtered[['ra', 'dec', 'mu_class', 'flags', 'flux_radius', 'elongation', 'mag_auto']].isna().any(axis=1)
filtered.loc[mask, 'catalog_reject_reason'] += 'missing_catalog_value;'

filtered['catalog_status'] = np.where(filtered['catalog_reject_reason'] == '', 'accept', 'reject')

print('Catalog-stage accepted:', int((filtered['catalog_status'] == 'accept').sum()))
print('Catalog-stage rejected:', int((filtered['catalog_status'] == 'reject').sum()))
print('
Top catalog rejection reasons:')
print(filtered.loc[filtered['catalog_status'] == 'reject', 'catalog_reject_reason'].value_counts().head(20))

display(filtered.head())

## Cell 5 — Match accepted manifest rows to manually downloaded FITS files

In [ ]:
available_files = {fp.name: fp for fp in RAW_DIR.glob('*.fits')}
available_files.update({fp.name: fp for fp in RAW_DIR.glob('*.fit')})

candidate_df = filtered[filtered['catalog_status'] == 'accept'].copy()
candidate_df['file_exists'] = candidate_df['filename'].isin(available_files)

matched_df = candidate_df[candidate_df['file_exists']].copy().reset_index(drop=True)
missing_df = candidate_df[~candidate_df['file_exists']].copy().reset_index(drop=True)

print('Accepted in catalog stage :', len(candidate_df))
print('Matched local FITS files  :', len(matched_df))
print('Missing local FITS files  :', len(missing_df))

display(matched_df.head())

## Cell 6 — Helper functions for FITS reading, pixel scale, normalisation, and image QA

In [ ]:
def get_pixel_scale_arcsec(header, default=DEFAULT_PIXEL_SCALE_ARCSEC):
    try:
        w = WCS(header)
        if not w.has_celestial:
            return default
        scales = proj_plane_pixel_scales(w.celestial) * 3600.0
        scales = np.asarray(scales, dtype=float)
        if np.all(np.isfinite(scales)) and np.all(scales > 0):
            return float(np.mean(scales))
    except Exception:
        pass
    return default


def robust_normalize(img):
    img = np.array(img, dtype=np.float32)
    finite = np.isfinite(img)
    finite_fraction = float(finite.mean())
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

    sky_level = float(np.median(img))
    img_sub = img - sky_level

    p99_flux = float(np.percentile(img_sub, 99.0))
    scale = p99_flux if p99_flux > 0 else 1.0

    norm = (img_sub / scale).astype(np.float32)
    norm = np.clip(norm, -1.0, 5.0)
    return norm, sky_level, p99_flux, finite_fraction


def edge_abs_mean(arr):
    edge = np.concatenate([arr[0, :], arr[-1, :], arr[:, 0], arr[:, -1]])
    return float(np.mean(np.abs(edge)))


def image_level_qa(norm, finite_fraction, filename):
    reasons = []
    std = float(np.std(norm))
    peak = float(np.max(norm))
    edge_mean = edge_abs_mean(norm)

    if filename in REJECT_BY_FILENAME:
        reasons.append('manual_filename_reject')
    if finite_fraction < MIN_FINITE_FRACTION:
        reasons.append('too_many_nonfinite_pixels')
    if std < MIN_STD:
        reasons.append('near_empty_or_too_flat')
    if peak > MAX_PEAK_AFTER_NORM:
        reasons.append('extreme_peak_possible_star_or_artifact')
    if edge_mean > MAX_EDGE_ABS_MEAN:
        reasons.append('strong_edge_signal_possible_bad_crop')

    return reasons, {
        'std_norm': std,
        'peak_norm': peak,
        'edge_abs_mean': edge_mean,
        'finite_fraction': finite_fraction,
    }

## Cell 7 — Build the final accepted sample after image-level QA

This stage combines:
1. **catalog-based rejection**, then
2. **image-level QA**.

In [ ]:
manifest_rows = []
image_store = {}

for _, row in tqdm(matched_df.iterrows(), total=len(matched_df), desc='Processing matched FITS'):
    fp = available_files[row['filename']]

    try:
        with fits.open(fp, memmap=False) as hdul:
            data = hdul[0].data
            header = hdul[0].header

        if data is None:
            manifest_rows.append({**row.to_dict(), 'final_status': 'reject', 'final_reason': 'empty_primary_hdu'})
            continue

        data = np.squeeze(np.array(data))
        if data.ndim != 2:
            manifest_rows.append({**row.to_dict(), 'final_status': 'reject', 'final_reason': f'non_2d_shape_{data.shape}'})
            continue

        pixel_scale = get_pixel_scale_arcsec(header)
        norm, sky_level, p99_flux, finite_fraction = robust_normalize(data)
        qa_reasons, qa_stats = image_level_qa(norm, finite_fraction, row['filename'])

        final_status = 'accept' if len(qa_reasons) == 0 else 'reject'
        final_reason = 'passed_catalog_and_image_qa' if len(qa_reasons) == 0 else ';'.join(qa_reasons)

        rec = {
            **row.to_dict(),
            'shape_y': int(data.shape[0]),
            'shape_x': int(data.shape[1]),
            'pixel_scale_arcsec': float(pixel_scale),
            'cutout_size_arcsec_y': float(data.shape[0] * pixel_scale),
            'cutout_size_arcsec_x': float(data.shape[1] * pixel_scale),
            'sky_level': float(sky_level),
            'p99_flux': float(p99_flux),
            'final_status': final_status,
            'final_reason': final_reason,
            **qa_stats,
        }
        manifest_rows.append(rec)
        image_store[row['filename']] = norm

    except Exception as e:
        manifest_rows.append({**row.to_dict(), 'final_status': 'reject', 'final_reason': f'exception: {type(e).__name__}: {e}'})

final_df = pd.DataFrame(manifest_rows)
accepted_df = final_df[final_df['final_status'] == 'accept'].copy().reset_index(drop=True)
rejected_df = final_df[final_df['final_status'] == 'reject'].copy().reset_index(drop=True)

print('Final accepted:', len(accepted_df))
print('Final rejected:', len(rejected_df))

display(accepted_df.head())

## Cell 8 — Export `.npy` files and metadata

In [ ]:
for old_file in NPY_DIR.glob('*.npy'):
    old_file.unlink()

metadata = []
for _, row in accepted_df.iterrows():
    filename = row['filename']
    arr = image_store[filename]
    stem = Path(filename).stem
    npy_name = f'{stem}.npy'
    np.save(NPY_DIR / npy_name, arr)

    metadata.append({
        'filename': filename,
        'npy_name': npy_name,
        'ra': float(row['ra']),
        'dec': float(row['dec']),
        'shape': [int(arr.shape[0]), int(arr.shape[1])],
        'mu_class': int(row['mu_class']),
        'flags': int(row['flags']),
        'flux_radius': float(row['flux_radius']),
        'elongation': float(row['elongation']),
        'mag_auto': float(row['mag_auto']),
        'pixel_scale_arcsec': float(row['pixel_scale_arcsec']),
        'cutout_size_arcsec': CUTOUT_SIZE_ARCSEC,
        'sky_level': float(row['sky_level']),
        'p99_flux': float(row['p99_flux']),
        'std_norm': float(row['std_norm']),
        'peak_norm': float(row['peak_norm']),
        'edge_abs_mean': float(row['edge_abs_mean']),
    })

with open(META_DIR / 'cutout_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

filtered.to_csv(META_DIR / 'catalog_stage_manifest.csv', index=False)
final_df.to_csv(META_DIR / 'final_manifest.csv', index=False)
accepted_df.to_csv(META_DIR / 'accepted_cutouts.csv', index=False)
rejected_df.to_csv(META_DIR / 'rejected_cutouts.csv', index=False)

print(f'Saved {len(metadata)} .npy files to: {NPY_DIR.resolve()}')
print(f'Metadata JSON written to: {(META_DIR / "cutout_metadata.json").resolve()}')

## Cell 9 — Verify `.npy` export

In [ ]:
saved_npy = sorted(NPY_DIR.glob('*.npy'))
print(f'Number of saved .npy files: {len(saved_npy)}')

for fp in saved_npy[:10]:
    arr = np.load(fp)
    print(fp.name, arr.shape, arr.dtype, float(arr.min()), float(arr.max()))

## Cell 10 — Make a grid JPG for visual review

In [ ]:
accepted_names = accepted_df['filename'].tolist()
num_accept = len(accepted_names)

if num_accept == 0:
    print('No accepted cutouts available for plotting.')
else:
    ncols = 10
    nrows = math.ceil(num_accept / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.1 * ncols, 2.1 * nrows), facecolor='#111111')

    axes = np.array(axes).reshape(nrows, ncols)
    for ax in axes.ravel():
        ax.set_facecolor('#111111')
        ax.axis('off')

    for i, name in enumerate(accepted_names):
        ax = axes.ravel()[i]
        ax.imshow(image_store[name], cmap='gray', origin='lower', vmin=-0.2, vmax=1.0)
        ax.set_title(f'#{i:02d}', color='white', fontsize=8)
        ax.axis('off')

    fig.suptitle(
        f'COSMOS HST/ACS F814W — {num_accept} accepted manual cutouts with manifest ({CUTOUT_SIZE_ARCSEC:.1f}" × {CUTOUT_SIZE_ARCSEC:.1f}")',
        color='white', fontsize=16
    )
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    grid_path = PLOT_DIR / 'galaxy_grid.jpg'
    fig.savefig(grid_path, dpi=180, facecolor=fig.get_facecolor(), bbox_inches='tight')
    plt.show()
    print('Saved grid to:', grid_path.resolve())

## Cell 11 — Notes

### What is a manifest?
A **manifest** is just a bookkeeping table that links each local FITS filename to its catalog properties.

### What is catalog-based rejection?
It means rejecting files using trusted survey catalog measurements rather than only eyeballing the cutout image.

### Why use both catalog and image QA?
- Catalog rules are strong for star/galaxy selection and detection quality.
- Image QA is strong for artifacts, empty fields, bad crops, or suspicious brightness structure.
- Manual review is still useful for borderline cases.